In [2]:
##Apply a Deep Learning classification model on the following data and generate predictions and prediction probs?

data = {
    'Age': [57, 36, 41, 39, 64, 29, 33, 41, 22, 55],
    'Income': [35841, 60757, 86578, 89250, 52337, 78232, 40443, 96221, 83296, 80993],
    'Gender': ['Female', 'Male', 'Female', 'Female', 'Male', 'Female', 'Female', 'Female', 'Female', 'Female'],
    'Education': ['PhD', 'Masters', 'Bachelors', 'Masters', 'Bachelors', 'Bachelors', 'Masters', 'Bachelors', 'High School', 'PhD'],
    'Target': ['No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes']
}

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler,OrdinalEncoder,OneHotEncoder
from keras import regularizers
from sklearn.metrics import classification_report

import pandas as pd
df=pd.DataFrame(data)
df.info()

#filter the categorical and numerical columns
categorical=df.select_dtypes(include=['object']).columns
numerical=df.select_dtypes(include=['int64', 'float64']).columns
print("Categorical columns:", categorical)
print("Numerical columns:", numerical)
#Encode the categorical columns
def transform(df):
    for col in categorical:
        le=LabelEncoder()
        df[col]=le.fit_transform(df[col])
        print(df.head())
    return df
#Second Method
def transform2(df):
    new_transform=OrdinalEncoder()
    df[categorical]=new_transform.fit_transform(df[categorical])
    print(df)
    return df
#Third Method
def transform3(df):
    ohe=OneHotEncoder(sparse_output=False)
    ohe_df=pd.DataFrame(ohe.fit_transform(df[categorical]),columns=ohe.get_feature_names_out(categorical))
    df=df.drop(columns=categorical,axis=1)
    df=pd.concat([df,ohe_df],axis=1)
    print(df)
    return df

# df_categorical=transform(df)
df_categorical=transform2(df)
# df_categorical=transform3(df)

#scale the numerical features
def scale_numerical_features(df):
    scaler=StandardScaler()
    df[numerical]=scaler.fit_transform(df[numerical])
    print(df)
    return df

df_numerical=scale_numerical_features(df_categorical)
X_train=pd.concat([df_numerical,df_categorical],axis=1)
X=X_train.drop(columns=['Target'],axis=1)
y=df_numerical['Target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

import keras
model=keras.Sequential()
model.add(keras.layers.Input(name="input_layer", shape=(X_train.shape[1],)))
model.add(keras.layers.Dense(name="dense_64",units=64, activation='relu',kernel_regularizer=regularizers.l1(0.01)))
model.add(keras.layers.Dense(name="dense_32",units=32, activation='relu',kernel_regularizer=regularizers.l2(0.01)   ))
model.add(keras.layers.Dense(name="dense_output", units=1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=100,batch_size=4, validation_split=0.2)
y_test_predict=model.predict(X_test)
y_train_predict=model.predict(X_train)
from sklearn.metrics import accuracy_score
import numpy as np

test_accuracy=accuracy_score(y_true=y_test, y_pred=np.argmax(y_test_predict,axis=1))
train_accuracy=accuracy_score(y_true=y_train, y_pred=np.argmax(y_train_predict,axis=1))
print("Test accuracy:", test_accuracy)
print("Train accuracy:", train_accuracy)    


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Age        10 non-null     int64 
 1   Income     10 non-null     int64 
 2   Gender     10 non-null     object
 3   Education  10 non-null     object
 4   Target     10 non-null     object
dtypes: int64(2), object(3)
memory usage: 532.0+ bytes
Categorical columns: Index(['Gender', 'Education', 'Target'], dtype='object')
Numerical columns: Index(['Age', 'Income'], dtype='object')
   Age  Income  Gender  Education  Target
0   57   35841     0.0        3.0     0.0
1   36   60757     1.0        2.0     1.0
2   41   86578     0.0        0.0     1.0
3   39   89250     0.0        2.0     0.0
4   64   52337     1.0        0.0     1.0
5   29   78232     0.0        0.0     0.0
6   33   40443     0.0        2.0     1.0
7   41   96221     0.0        0.0     0.0
8   22   83296     0.0        1.0     1.0
9   55   8099

In [4]:
X=df.iloc[:,0:4]
y=df.iloc[:,4]
y

0     No
1    Yes
2    Yes
3     No
4    Yes
5     No
6    Yes
7     No
8    Yes
9    Yes
Name: Target, dtype: object